# Document Validation POC — Invoice & Order Form Extraction with Ollama

End-to-end recreation of a GenAI document validation system.

**Pipeline:**
1. **Text extraction** — pull text + tables from PDF
2. **Field extraction** — local LLM (Ollama) returns structured JSON
3. **Master-data validation** — rule-based checks (totals, dates, GST math, ID format)
4. **RAG anomaly detection** — embed historical invoices, flag unusual patterns
5. **Human-review report** — summary of flags + confidence per field

**What you need:**
- Ollama running locally (`ollama serve`)
- A model pulled, e.g. `ollama pull llama3.1:8b` (or `mistral`, `qwen2.5`, etc.)
- An embedding model: `ollama pull nomic-embed-text`
- Python 3.10+


## Stage 0 — Install dependencies & verify Ollama

In [ ]:
# Run once. Skip if already installed.
!pip install -q pdfplumber pypdf requests pandas numpy scikit-learn


In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434"
GEN_MODEL = "llama3.1:8b"          # change to whatever you have pulled
EMBED_MODEL = "nomic-embed-text"   # for the RAG layer

# Sanity-check that Ollama is up and the models are available
def check_ollama():
    try:
        r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
        models = [m["name"] for m in r.json().get("models", [])]
        print("Ollama is up. Installed models:")
        for m in models:
            print(f"  - {m}")
        for required in (GEN_MODEL, EMBED_MODEL):
            if not any(required in m for m in models):
                print(f"\n[!] '{required}' not found. Run: ollama pull {required}")
        return models
    except Exception as e:
        print(f"[!] Cannot reach Ollama at {OLLAMA_URL}: {e}")
        print("Start it with `ollama serve` in a separate terminal.")
        return []

check_ollama()


## Stage 1 — Extract raw text from the PDF

Two passes: (a) full-page text via pdfplumber, (b) tables as structured rows. The LLM gets both.


In [ ]:
import pdfplumber
from pathlib import Path

PDF_PATH = "C:\\Users\\Mateti sai pranay\\Downloads\\.net\\python\\re_vamp\\pdf-text_pipline\\docs_images\\invoices\\medical_similar\\shiva\\invoice1.pdf"   # <-- put your PDF next to this notebook

def extract_pdf(path):
    """Return (full_text, list_of_tables) from a PDF."""
    full_text_parts = []
    all_tables = []
    with pdfplumber.open(path) as pdf:
        for page_num, page in enumerate(pdf.pages, 1):
            text = page.extract_text() or ""
            full_text_parts.append(f"--- PAGE {page_num} ---\n{text}")
            for tbl in page.extract_tables() or []:
                all_tables.append({"page": page_num, "rows": tbl})
    return "\n\n".join(full_text_parts), all_tables

if Path(PDF_PATH).exists():
    text, tables = extract_pdf(PDF_PATH)
    print(f"Extracted {len(text)} chars of text and {len(tables)} table(s).")
    print("\n--- TEXT PREVIEW ---")
    print(text[:1500])
else:
    print(f"[!] {PDF_PATH} not found. Place your invoice PDF there.")
    text, tables = "", []


In [ ]:
# Inspect the tables pdfplumber found (the line-item grid is usually here)
for i, t in enumerate(tables):
    print(f"\n=== Table {i+1} (page {t['page']}, {len(t['rows'])} rows) ===")
    for row in t["rows"][:5]:
        print(row)
    if len(t["rows"]) > 5:
        print(f"... +{len(t['rows'])-5} more rows")


## Stage 2 — Field extraction with Ollama

We send the raw text + tables to the LLM with a strict JSON schema and a low temperature. The schema mirrors what a Form 222 / invoice validator needs: seller, buyer, invoice metadata, line items, tax breakdown, totals.


In [ ]:
import json

def ollama_generate(prompt, model=GEN_MODEL, temperature=0.0, json_mode=True):
    """Single-turn completion. json_mode=True forces valid JSON output."""
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature, "num_ctx": 8192},
    }
    if json_mode:
        payload["format"] = "json"
    r = requests.post(f"{OLLAMA_URL}/api/generate", json=payload, timeout=300)
    r.raise_for_status()
    return r.json()["response"]

# Quick smoke test
print(ollama_generate('Reply with JSON: {"ok": true}'))


In [ ]:
EXTRACTION_SCHEMA = {
    "seller": {
        "name": "", "address": "", "phone": "",
        "gstin": "", "drug_license_no": ""
    },
    "buyer": {
        "name": "", "address": "", "phone": "",
        "state_code": "", "drug_license_no": ""
    },
    "invoice": {
        "number": "", "date": "", "due_date": "",
        "type": "",          # CREDIT / CASH
        "lr_no": "", "lr_date": "",
        "order_no": "", "order_date": "",
        "transport": "", "cases": 0
    },
    "line_items": [
        {
            "sl_no": 0, "hsn": "", "mfr": "", "product_name": "",
            "pack": "", "batch": "", "expiry": "",
            "qty": 0.0, "free": 0.0,
            "mrp": 0.0, "rate": 0.0, "discount_pct": 0.0,
            "sgst_pct": 0.0, "cgst_pct": 0.0,
            "amount": 0.0, "net_amount": 0.0
        }
    ],
    "tax_summary": {
        "by_slab": [
            {"slab_pct": 0.0, "taxable": 0.0, "sgst": 0.0, "cgst": 0.0, "total_gst": 0.0}
        ],
        "total_taxable": 0.0,
        "total_sgst": 0.0,
        "total_cgst": 0.0,
        "total_gst": 0.0
    },
    "totals": {
        "items_count": 0,
        "qty_total": 0.0,
        "discount_total": 0.0,
        "freight": 0.0,
        "cr_dr_note": 0.0,
        "grand_total": 0.0,
        "grand_total_in_words": ""
    }
}

EXTRACTION_PROMPT = """You are extracting fields from a GST invoice.
Return ONE JSON object exactly matching this schema. Use null for missing values.
Numbers must be numbers, not strings. Dates as YYYY-MM-DD when possible.

SCHEMA:
{schema}

INVOICE TEXT:
{text}

EXTRACTED TABLES (raw rows from PDF):
{tables}

Return ONLY the JSON, no commentary."""

def extract_fields(text, tables):
    prompt = EXTRACTION_PROMPT.format(
        schema=json.dumps(EXTRACTION_SCHEMA, indent=2),
        text=text[:6000],
        tables=json.dumps([t["rows"] for t in tables], indent=2)[:4000],
    )
    raw = ollama_generate(prompt)
    return json.loads(raw)

extracted = extract_fields(text, tables)
print(json.dumps(extracted, indent=2)[:3000])


## Stage 3 — Rule-based validation against master data + math checks

Three categories of checks:
- **Format checks** — GSTIN regex, date sanity, expiry not in past
- **Math checks** — line `qty × rate = amount`, sum of lines = subtotal, GST math
- **Master-data checks** — does the seller GSTIN exist in our supplier master? Buyer license valid?

In a real system, master data lives in a database. Here we use small dicts you can swap for a DB call.


In [ ]:
import re
from datetime import datetime

# --- Master data (replace with DB queries in production) ---
SUPPLIER_MASTER = {
    "36AAOFS3456N1ZG": {
        "name": "SRI SHIVA SAI POULTRY & VETERINARY NEEDS",
        "state": "TELANGANA",
        "active": True,
        "drug_license": "TG/22/02/2001-20590,20591,20592,593",
    },
}

BUYER_MASTER = {
    "TS/KGM/2023-106432": {
        "name": "BHARGAV MEDICAL STORES",
        "active": True,
        "state": "TELANGANA",
    },
}

GSTIN_REGEX = re.compile(r"^[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z]{1}[1-9A-Z]{1}Z[0-9A-Z]{1}$")


def validate(doc):
    """Return list of {severity, code, message, field} flags."""
    flags = []

    # ---- Format checks ----
    gstin = (doc.get("seller") or {}).get("gstin", "") or ""
    if gstin and not GSTIN_REGEX.match(gstin):
        flags.append({"severity": "high", "code": "GSTIN_FORMAT",
                      "field": "seller.gstin",
                      "message": f"GSTIN '{gstin}' fails format check."})

    # ---- Master-data checks ----
    if gstin and gstin not in SUPPLIER_MASTER:
        flags.append({"severity": "high", "code": "SUPPLIER_NOT_FOUND",
                      "field": "seller.gstin",
                      "message": f"Supplier GSTIN '{gstin}' not in master."})
    elif gstin in SUPPLIER_MASTER:
        master = SUPPLIER_MASTER[gstin]
        if not master["active"]:
            flags.append({"severity": "high", "code": "SUPPLIER_INACTIVE",
                          "field": "seller.gstin",
                          "message": "Supplier is marked inactive."})
        extracted_name = (doc.get("seller") or {}).get("name", "") or ""
        if master["name"].lower() not in extracted_name.lower():
            flags.append({"severity": "medium", "code": "SUPPLIER_NAME_MISMATCH",
                          "field": "seller.name",
                          "message": f"Seller name '{extracted_name}' "
                                     f"doesn't match master '{master['name']}'."})

    buyer_lic = (doc.get("buyer") or {}).get("drug_license_no", "") or ""
    if buyer_lic and buyer_lic not in BUYER_MASTER:
        flags.append({"severity": "medium", "code": "BUYER_NOT_FOUND",
                      "field": "buyer.drug_license_no",
                      "message": f"Buyer license '{buyer_lic}' not in master."})

    # ---- Date checks ----
    today = datetime.today().date()
    for li in doc.get("line_items", []) or []:
        exp = li.get("expiry")
        if exp:
            try:
                # Accept formats like '12/24', '2024-12', '2024-12-01'
                if re.match(r"^\d{2}/\d{2}$", exp):
                    mm, yy = exp.split("/")
                    exp_date = datetime(2000 + int(yy), int(mm), 1).date()
                else:
                    exp_date = datetime.fromisoformat(exp[:10]).date()
                if exp_date < today:
                    flags.append({"severity": "high", "code": "EXPIRED_PRODUCT",
                                  "field": f"line_items[{li.get('sl_no')}].expiry",
                                  "message": f"{li.get('product_name')} expired on {exp_date}."})
            except Exception:
                flags.append({"severity": "low", "code": "DATE_PARSE",
                              "field": f"line_items[{li.get('sl_no')}].expiry",
                              "message": f"Could not parse expiry '{exp}'."})

    # ---- Math checks (tolerance for rupee rounding) ----
    TOL = 1.0
    for li in doc.get("line_items", []) or []:
        qty = li.get("qty") or 0
        rate = li.get("rate") or 0
        amt = li.get("amount") or 0
        expected = round(qty * rate, 2)
        if abs(expected - amt) > TOL:
            flags.append({"severity": "medium", "code": "LINE_AMOUNT_MISMATCH",
                          "field": f"line_items[{li.get('sl_no')}].amount",
                          "message": f"qty*rate={expected} but amount={amt}."})

    # Sum of line nets vs grand total (rough; ignores freight/discount nuances)
    sum_net = sum((li.get("net_amount") or 0) for li in doc.get("line_items", []) or [])
    grand = (doc.get("totals") or {}).get("grand_total") or 0
    if grand and abs(sum_net - grand) > 5.0:
        flags.append({"severity": "low", "code": "GRAND_TOTAL_DRIFT",
                      "field": "totals.grand_total",
                      "message": f"Sum of net_amount={sum_net:.2f} vs grand_total={grand:.2f}."})

    # ---- Items count check ----
    declared_count = (doc.get("totals") or {}).get("items_count") or 0
    actual_count = len(doc.get("line_items", []) or [])
    if declared_count and declared_count != actual_count:
        flags.append({"severity": "medium", "code": "ITEM_COUNT_MISMATCH",
                      "field": "totals.items_count",
                      "message": f"Declared {declared_count} items but {actual_count} extracted."})

    return flags

flags = validate(extracted)
print(f"{len(flags)} flag(s) raised.")
for f in flags:
    print(f"  [{f['severity'].upper()}] {f['code']} @ {f['field']}: {f['message']}")


## Stage 4 — RAG-based anomaly detection

Idea: keep a small corpus of past invoices and their known-good patterns. Embed the new invoice's structured summary and find the most similar historical ones. Flag big deviations in price, quantity, or vendor pairs.

For the demo we seed a tiny corpus. In production this would be a vector DB (Chroma, pgvector, FAISS).


In [ ]:
import numpy as np
from numpy.linalg import norm

def embed(text, model=EMBED_MODEL):
    r = requests.post(f"{OLLAMA_URL}/api/embeddings",
                      json={"model": model, "prompt": text}, timeout=120)
    r.raise_for_status()
    return np.array(r.json()["embedding"], dtype=np.float32)

# Tiny historical corpus (replace with real history)
HISTORY = [
    {"id": "H001",
     "summary": "Supplier 36AAOFS3456N1ZG sold ZENCLONEX-I 1LIT to BHARGAV MEDICAL STORES at rate 1100",
     "rate": 1100.0, "product": "ZENCLONEX-I"},
    {"id": "H002",
     "summary": "Supplier 36AAOFS3456N1ZG sold MORAL INJ 30ML to BHARGAV MEDICAL STORES at rate 600",
     "rate": 600.0, "product": "MORAL INJ"},
    {"id": "H003",
     "summary": "Supplier 36AAOFS3456N1ZG sold ULTROX LIQ 100ML to BHARGAV MEDICAL STORES at rate 160",
     "rate": 160.0, "product": "ULTROX LIQ"},
]

# Embed history once (cache in real life)
for h in HISTORY:
    h["vec"] = embed(h["summary"])
print(f"Indexed {len(HISTORY)} historical invoices.")


In [ ]:
def cosine(a, b):
    return float(np.dot(a, b) / (norm(a) * norm(b) + 1e-9))

def line_summary(seller_gstin, buyer_name, li):
    return (f"Supplier {seller_gstin} sold {li.get('product_name')} "
            f"to {buyer_name} at rate {li.get('rate')}")

def rag_anomalies(doc, history, price_tolerance=0.20):
    """For each line item, find the closest historical line and flag price drift."""
    seller = (doc.get("seller") or {}).get("gstin", "")
    buyer = (doc.get("buyer") or {}).get("name", "")
    anomalies = []
    for li in doc.get("line_items", []) or []:
        query = line_summary(seller, buyer, li)
        qvec = embed(query)
        sims = [(cosine(qvec, h["vec"]), h) for h in history]
        sims.sort(key=lambda x: x[0], reverse=True)
        best_score, best = sims[0]
        rate = li.get("rate") or 0
        if best_score > 0.75 and best["rate"] > 0:
            drift = abs(rate - best["rate"]) / best["rate"]
            if drift > price_tolerance:
                anomalies.append({
                    "severity": "medium",
                    "code": "PRICE_DRIFT",
                    "field": f"line_items[{li.get('sl_no')}].rate",
                    "message": (f"{li.get('product_name')} rate {rate} differs "
                                f"by {drift:.0%} from historical {best['rate']} "
                                f"(matched {best['id']}, sim={best_score:.2f})."),
                })
    return anomalies

rag_flags = rag_anomalies(extracted, HISTORY)
print(f"{len(rag_flags)} RAG anomalies detected.")
for f in rag_flags:
    print(f"  [{f['severity'].upper()}] {f['code']}: {f['message']}")


## Stage 5 — Combined human-review report

One verdict per document: PASS, REVIEW, or BLOCK based on flag severities.


In [ ]:
def make_report(doc, rule_flags, rag_flags):
    all_flags = rule_flags + rag_flags
    high = [f for f in all_flags if f["severity"] == "high"]
    medium = [f for f in all_flags if f["severity"] == "medium"]
    low = [f for f in all_flags if f["severity"] == "low"]

    if high:
        verdict = "BLOCK"
    elif medium:
        verdict = "REVIEW"
    else:
        verdict = "PASS"

    return {
        "invoice_number": (doc.get("invoice") or {}).get("number"),
        "seller": (doc.get("seller") or {}).get("name"),
        "buyer": (doc.get("buyer") or {}).get("name"),
        "grand_total": (doc.get("totals") or {}).get("grand_total"),
        "verdict": verdict,
        "summary": {"high": len(high), "medium": len(medium), "low": len(low)},
        "flags": all_flags,
    }

report = make_report(extracted, flags, rag_flags)
print(json.dumps(report, indent=2))


In [ ]:
# Pretty-print as a review queue row
print(f"""
================================================================
INVOICE {report['invoice_number']}    VERDICT: {report['verdict']}
----------------------------------------------------------------
Seller : {report['seller']}
Buyer  : {report['buyer']}
Total  : {report['grand_total']}
Flags  : {report['summary']['high']} high | {report['summary']['medium']} medium | {report['summary']['low']} low
================================================================
""")
for f in report["flags"]:
    print(f"  [{f['severity'].upper():6}] {f['code']:25} {f['message']}")


## Next steps

- **Swap the master-data dicts** for queries against your real DB (Postgres, etc.).
- **Replace the in-memory `HISTORY` list** with a vector store — Chroma is the easiest drop-in:
  ```python
  import chromadb
  client = chromadb.PersistentClient(path="./vec_store")
  col = client.get_or_create_collection("invoices")
  ```
- **Add a confidence score** per field by re-prompting the LLM with `"Return JSON with each field as {value, confidence: 0..1}"`.
- **Batch processing**: wrap the whole pipeline in `process(pdf_path)` and run over a folder.
- **Compliance rules layer**: encode your DEA/GST rules as Python predicates that consume the extracted JSON and emit flags. The same flag schema flows into the report.
- **Model choice**: `llama3.1:8b` works for this; `qwen2.5:7b` is often better at strict JSON. For larger invoices, bump `num_ctx` to 16384 or use `llama3.1:70b` if your machine can handle it.
